# EDA — Mix energetico in Europa: nucleare, rinnovabili e fossili

**Obiettivo del notebook**

Esplorare come è cambiato nel tempo il mix di fonti usate per produrre elettricità nei paesi europei,
con un focus particolare sul confronto tra **nucleare**, **rinnovabili** e **fonti fossili**.

## 1. Fonte dei dati

I dati usati in questo notebook provengono dal dataset **Energy Data** di
[**Our World in Data (OWID)**](https://github.com/owid/energy-data), un progetto di ricerca open-access
dell'Università di Oxford.

- OWID non raccoglie i dati in prima persona, ma li **aggrega e armonizza** da fonti istituzionali
  riconosciute a livello internazionale, principalmente:
  - **Ember** — *Yearly Electricity Data*, dati elettrici annuali (la fonte principale per l'elettricità dal 2000 in poi,
    e dal 1990 per i paesi europei)
  - **Energy Institute** — *Statistical Review of World Energy* (fonte storica, usata per gli anni precedenti)
  - Dati di popolazione e PIL da fonti demografiche/economiche standard usate da OWID
- Il dataset è **aggiornato regolarmente** (ultimo aggiornamento verificato: 2026) ed è openly licensed
  (**CC BY** — riutilizzabile citando la fonte)
- È lo stesso dataset usato per generare i grafici pubblicati su https://ourworldindata.org/energy

**Link diretti:**
- Repository GitHub: https://github.com/owid/energy-data
- File CSV completo: `owid-energy-data.csv` (nella root del repository)
- Codebook (descrizione di ogni colonna): `owid-energy-codebook.csv`
- Pagina di presentazione: https://ourworldindata.org/energy

**Citazione consigliata** (come richiesto da OWID per l'uso dei dati):
> Ember (2026); Energy Institute - Statistical Review of World Energy (2025) — with major processing by Our World in Data.

In [86]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path.cwd()
df = pd.read_csv(DATA_DIR / "data" / "owid-energy-data.csv")

print(df.shape)
df.head()

(23377, 130)


,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
3,ASEAN (Ember),2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
4,ASEAN (Ember),2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN


In [87]:
codebook = pd.read_csv(DATA_DIR / "data" / "owid-energy-codebook.csv")

print(codebook.shape)
codebook.head()

(130, 5)


,column,title,description,unit,source
0,country,Country,Geographic location.,NaN,Our World in Data - Regions (2025)
1,year,Year,Year of observation.,NaN,Our World in Data - Regions (2025)
2,iso_code,ISO code,ISO 3166-1 alpha-3 three-letter country codes.,NaN,International Organization for Standardization...
3,population,Population,"Population by country, available from 10,000 B...",people,Population based on various sources (2024) [ht...
4,gdp,Gross domestic product (GDP),Total economic output of a country or region p...,international-$ in 2011 prices ($),Bolt and van Zanden – Maddison Project Databas...


## 2. Struttura del dataset

Il file è in formato **"long"**: una riga per ogni combinazione di *paese* e *anno*.
Prima di tutto controlliamo dimensioni, periodo temporale coperto e quante "entità" (paesi/aggregati)
sono presenti.


In [88]:
print(f"Righe: {df.shape[0]:,}")
print(f"Colonne: {df.shape[1]}")
print(f"Periodo: {df['year'].min()} - {df['year'].max()}")
print(f"Entità (paesi + aggregati): {df['country'].nunique()}")


Righe: 23,377
Colonne: 130
Periodo: 1900 - 2025
Entità (paesi + aggregati): 314


**Nota importante:** la colonna `country` non contiene *solo* paesi. OWID include anche
**aggregati** (continenti, "World", gruppi di reddito, organizzazioni come "EU (Ember)", "G7 (Ember)" ecc.)
per permettere confronti a livello macro nei propri grafici.

Questi aggregati **non hanno un `iso_code`** (il codice ISO a 3 lettere dei paesi), quindi possiamo
usare questo campo per individuarli ed escluderli quando vogliamo lavorare solo su paesi reali.


In [89]:
# Quante "entità" non sono paesi veri (non hanno iso_code)?
aggregati = sorted(df.loc[df["iso_code"].isna(), "country"].unique())
print(f"Aggregati/non-paesi individuati: {len(aggregati)}")
aggregati[:15]  # primi esempi


Aggregati/non-paesi individuati: 94


['ASEAN (Ember)',
 'Africa',
 'Africa (EI)',
 'Africa (EIA)',
 'Africa (Ember)',
 'Africa (Shift)',
 'Asia',
 'Asia (Ember)',
 'Asia Pacific (EI)',
 'Asia and Oceania (EIA)',
 'Asia and Oceania (Shift)',
 'Australia and New Zealand (EIA)',
 'CIS (EI)',
 'Central America (EI)',
 'Central and South America (EIA)']

## 3. Analisi dei valori mancanti

Vogliamo capire se i `NaN` nei paesi europei sono **strutturali** (gap sparsi nella serie) oppure semplicemente **left-censored**: ogni paese inizia a registrare dati in un anno diverso, e prima di quel momento i valori sono `NaN` per definizione.

Le due situazioni hanno implicazioni molto diverse:
- **Left-censored**: il paese non aveva ancora dati disponibili → i `NaN` sono tutti concentrati nelle righe iniziali e scompaiono una volta che la serie parte.
- **Gap strutturale**: dati mancanti sparsi nel mezzo della serie → possibile per paesi con storie politiche particolari (es. ex-URSS, ex-Jugoslavia) o per variabili non rilevate sistematicamente.

La strategia: per ogni paese e ogni colonna chiave, verifico se tutti i `NaN` cadono *prima* del primo valore valido (`first_valid_year`). Se sì, il pattern è left-censored; altrimenti c'è almeno un gap interno.

In [90]:
EUROPE_ISO = {
    "ALB", "AND", "AUT", "BLR", "BEL", "BIH", "BGR", "HRV", "CYP", "CZE",
    "DNK", "EST", "FIN", "FRA", "DEU", "GRC", "HUN", "ISL", "IRL", "ITA",
    "XKX", "LVA", "LIE", "LTU", "LUX", "MLT", "MDA", "MCO", "MNE", "NLD",
    "MKD", "NOR", "POL", "PRT", "ROU", "RUS", "SMR", "SRB", "SVK", "SVN",
    "ESP", "SWE", "CHE", "UKR", "GBR", "VAT",
}

KEY_COLS = [
    "electricity_generation",
    "fossil_electricity",
    "nuclear_electricity",
    "renewables_electricity",
    "solar_electricity",
    "wind_electricity",
    "hydro_electricity",
]

df_eu = df[df["iso_code"].isin(EUROPE_ISO)].copy()
print(f"Paesi europei nel dataset: {df_eu['country'].nunique()}")
print(f"Righe totali: {len(df_eu):,}  |  Anni: {df_eu['year'].min()}–{df_eu['year'].max()}")

Paesi europei nel dataset: 40
Righe totali: 3,488  |  Anni: 1900–2025


In [ ]:
def classify_missing(series: pd.Series) -> dict:
    first_idx = series.first_valid_index()
    if first_idx is None:
        return {"first_valid": None, "nan_before": int(series.isna().sum()), "nan_after": 0, "pattern": "always-null"}
    nan_before = int(series.loc[:first_idx].iloc[:-1].isna().sum())
    nan_after  = int(series.loc[first_idx:].isna().sum())
    if nan_after == 0:
        pattern = "complete" if nan_before == 0 else "left-censored"
    else:
        pattern = "internal-gaps"
    return {"first_valid": first_idx, "nan_before": nan_before, "nan_after": nan_after, "pattern": pattern}


def gap_ranges(s: pd.Series, first_idx) -> str | None:
    """Restituisce gli anni NaN dopo first_idx come stringa di intervalli, o None se assenti."""
    gap_years = s.loc[first_idx:][s.loc[first_idx:].isna()].index.tolist()
    if not gap_years:
        return None
    ranges, start, prev = [], gap_years[0], gap_years[0]
    for y in gap_years[1:]:
        if y == prev + 1:
            prev = y
        else:
            ranges.append(f"{start}–{prev}" if start != prev else str(start))
            start = prev = y
    ranges.append(f"{start}–{prev}" if start != prev else str(start))
    return ", ".join(ranges)


def build_miss(df_subset: pd.DataFrame) -> pd.DataFrame:
    """Classifica il pattern dei missing values per ogni coppia paese × colonna chiave."""
    records = []
    for country, grp in df_subset.groupby("country"):
        grp_sorted = grp.sort_values("year").set_index("year")
        for col in KEY_COLS:
            if col in grp_sorted.columns:
                records.append({"country": country, "column": col, **classify_missing(grp_sorted[col])})
    return pd.DataFrame(records)


miss = build_miss(df_eu)

In [ ]:
print("=== Distribuzione dei pattern — Europa (paese × colonna) ===")
print(miss["pattern"].value_counts().to_string())

print("\n=== Anno del primo dato valido ===")
pivot = miss[miss["pattern"] != "always-null"].pivot(
    index="country", columns="column", values="first_valid"
).astype("Int64")
print(pivot.to_string())

print("\n=== Gap interni ===")
for country, grp in df_eu.groupby("country"):
    grp_sorted = grp.sort_values("year").set_index("year")
    for col in KEY_COLS:
        if col not in grp_sorted.columns:
            continue
        s = grp_sorted[col]
        first_idx = s.first_valid_index()
        if first_idx is None:
            continue
        r = gap_ranges(s, first_idx)
        if r:
            print(f"  {country:30s}  {col:30s}  {r}")

## 4. Stessa analisi — resto del mondo

Ripetiamo l'analisi dei missing values sui paesi non europei (sempre escludendo gli aggregati senza `iso_code`). L'obiettivo è capire se il pattern left-censored / gap strutturale che abbiamo osservato per l'Europa è una caratteristica generale del dataset o se fuori dall'Europa la copertura dei dati è molto più frammentata.

In [ ]:
df_world = df[df["iso_code"].notna() & ~df["iso_code"].isin(EUROPE_ISO)].copy()
print(f"Paesi non europei nel dataset: {df_world['country'].nunique()}")

miss_w = build_miss(df_world)

print("\n=== Distribuzione dei pattern — Resto del mondo (paese × colonna) ===")
print(miss_w["pattern"].value_counts().to_string())

print("\n=== Anno del primo dato valido ===")
pivot_w = miss_w[miss_w["pattern"] != "always-null"].pivot(
    index="country", columns="column", values="first_valid"
).astype("Int64")
print(pivot_w.to_string())

print("\n=== Gap interni ===")
found_any = False
for country, grp in df_world.groupby("country"):
    grp_sorted = grp.sort_values("year").set_index("year")
    for col in KEY_COLS:
        if col not in grp_sorted.columns:
            continue
        s = grp_sorted[col]
        first_idx = s.first_valid_index()
        if first_idx is None:
            continue
        r = gap_ranges(s, first_idx)
        if r:
            found_any = True
            print(f"  {country:40s}  {col:30s}  {r}")
if not found_any:
    print("  Nessun gap interno.")

print("\n=== Casi always-null ===")
always_null_w = miss_w[miss_w["pattern"] == "always-null"][["country", "column"]]
print(always_null_w.sort_values(["country", "column"]).to_string(index=False))

In [97]:
LAST_YEAR = df["year"].max()  # 2025

def right_censored(group, col):
    """Restituisce l'ultimo anno con dato valido per una colonna, o None se sempre NaN."""
    s = group.sort_values("year").set_index("year")[col]
    last_idx = s.last_valid_index()
    return last_idx

# --- Europa ---
rc_eu = []
for country, grp in df_eu.groupby("country"):
    for col in KEY_COLS:
        if col not in grp.columns:
            continue
        last = right_censored(grp, col)
        if last is not None and last < LAST_YEAR:
            rc_eu.append({"country": country, "column": col, "last_valid": last, "missing_tail": LAST_YEAR - last})

rc_eu_df = pd.DataFrame(rc_eu).sort_values("missing_tail", ascending=False)

# --- Resto del mondo ---
rc_w = []
for country, grp in df_world.groupby("country"):
    for col in KEY_COLS:
        if col not in grp.columns:
            continue
        last = right_censored(grp, col)
        if last is not None and last < LAST_YEAR:
            rc_w.append({"country": country, "column": col, "last_valid": last, "missing_tail": LAST_YEAR - last})

rc_w_df = pd.DataFrame(rc_w).sort_values("missing_tail", ascending=False)

print(f"=== Right-censoring: EUROPA ({len(rc_eu_df)} coppie paese×colonna con serie troncata prima del {LAST_YEAR}) ===")
if rc_eu_df.empty:
    print("  Nessun caso.")
else:
    print(rc_eu_df.to_string(index=False))

print(f"\n=== Right-censoring: RESTO DEL MONDO ({len(rc_w_df)} coppie paese×colonna con serie troncata prima del {LAST_YEAR}) ===")
if rc_w_df.empty:
    print("  Nessun caso.")
else:
    print(rc_w_df.to_string(index=False))

=== Right-censoring: EUROPA (21 coppie paese×colonna con serie troncata prima del 2025) ===
country                 column  last_valid  missing_tail
Ukraine     fossil_electricity        2022             3
Ukraine       wind_electricity        2022             3
Ukraine      solar_electricity        2022             3
Ukraine      hydro_electricity        2022             3
Ukraine renewables_electricity        2022             3
Ukraine electricity_generation        2022             3
Ukraine    nuclear_electricity        2022             3
Albania      solar_electricity        2024             1
Albania     fossil_electricity        2024             1
Albania electricity_generation        2024             1
Albania renewables_electricity        2024             1
Albania    nuclear_electricity        2024             1
Albania       wind_electricity        2024             1
Iceland       wind_electricity        2024             1
Iceland      solar_electricity        2024           

### Conclusioni sull'analisi dei missing values

L'analisi ha permesso di classificare tutti i valori mancanti nelle variabili elettriche chiave. Il quadro emerso è netto: i `NaN` non sono casuali.

**Pattern dominante — left-censoring**
La grande maggioranza dei missing values, sia in Europa che nel resto del mondo, è concentrata agli inizi delle serie storiche. Ogni paese ha un anno di ingresso nel dataset che dipende da quando le rilevazioni statistiche hanno avuto inizio o sono diventate sistematiche. Prima di quell'anno i dati non esistono semplicemente perché non venivano raccolti, non perché siano andati persi. Si tratta quindi di **MNAR (Missing Not At Random)** con una causa pienamente osservabile: il tempo.

**Gap interni — discontinuità storiche**
I pochi casi di gap strutturali interni alla serie si concentrano su paesi che hanno attraversato fratture politiche significative (dissoluzione dell'URSS, frammentazione della Jugoslavia). Anche questi non sono random: la mancanza di dati è direttamente correlata a eventi storici documentati che hanno interrotto la continuità delle istituzioni statistiche nazionali.

**Always-null — copertura strutturalmente assente**
Fuori dall'Europa compaiono paesi per cui alcune variabili non hanno mai un valore nel dataset. Si tratta tipicamente di paesi molto piccoli o con capacità statistica limitata, per i quali determinate fonti (Ember, Energy Institute) non dispongono di rilevazioni. Anche in questo caso la mancanza è legata a caratteristiche strutturali del paese, non a errori di raccolta.

**Right-censoring**
L'analisi della coda destra ha rivelato che [*da completare dopo aver eseguito la cella precedente*].

**Implicazione per l'analisi**
Non essendoci missing values casuali, non è necessario ricorrere a tecniche di imputazione. La strategia corretta è restringere il perimetro temporale e geografico a dove i dati esistono — che per i principali paesi europei significa lavorare a partire dagli anni '90 — ed escludere esplicitamente le variabili o i paesi privi di copertura sufficiente.